# Hunyuan3D-2: Image-to-3D with Texture (RunPod)

Generate a textured 3D model from a single image using Tencent's Hunyuan3D-2.

**RunPod Setup:**
- GPU: RTX 4090 (24 GB VRAM)
- Template: RunPod PyTorch 2.x
- Container Disk: 20 GB (default)
- Volume Disk: **50 GB minimum** (Edit Pod > increase if needed)

**Storage note:** Model weights are ~15GB. The root `/` is only 20GB container disk.
This notebook symlinks `/root/.cache` → `/workspace/.cache` so all downloads go to the volume.

## Step 1: Verify GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected.")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"CUDA: {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")

## Step 2: Redirect all caches to /workspace

RunPod's root `/` is a small container disk (20GB). Model weights, pip cache, and temp files
must go to `/workspace` (your volume disk). This symlinks the entire cache directory and sets
environment variables so nothing writes to root.

In [ ]:
import os

# Symlink /root/.cache -> /workspace/.cache (the single most effective fix)
# This catches ALL cache writes: HuggingFace, pip, torch, etc.
!mkdir -p /workspace/.cache /workspace/tmp
!rsync -a /root/.cache/ /workspace/.cache/ 2>/dev/null || true
!rm -rf /root/.cache
!ln -sf /workspace/.cache /root/.cache

# Set environment variables as a safety net
os.environ["HF_HOME"] = "/workspace/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/workspace/.cache/huggingface/hub"
os.environ["HUGGINGFACE_HUB_CACHE"] = "/workspace/.cache/huggingface/hub"
os.environ["TRANSFORMERS_CACHE"] = "/workspace/.cache/huggingface/hub"
os.environ["TORCH_HOME"] = "/workspace/.cache/torch"
os.environ["PIP_CACHE_DIR"] = "/workspace/.cache/pip"
os.environ["TMPDIR"] = "/workspace/tmp"

# Verify
!df -h / /workspace
!echo ""
!ls -la /root/.cache
print("\nAll caches redirected to /workspace via symlink.")

## Step 3: Clone repo & install dependencies

In [ ]:
os.chdir("/workspace")
if not os.path.exists("/workspace/Hunyuan3D-2"):
    !git clone https://github.com/Tencent/Hunyuan3D-2.git
else:
    print("Repo already cloned.")
os.chdir("/workspace/Hunyuan3D-2")
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Step 1: Fix torchvision to match the pre-installed torch version
!python3 -c "import torch; print('torch:', torch.__version__)"
!pip install --no-cache-dir torchvision --index-url https://download.pytorch.org/whl/cu128 2>&1 | tail -3

# Step 2: Install remaining dependencies (never touch torch/torchvision again)
!pip install --no-cache-dir \
    "numpy==1.26.4" \
    "scipy==1.13.1" \
    "trimesh==4.4.9" \
    "transformers==4.44.2" \
    "diffusers==0.30.3" \
    "accelerate==0.34.2" \
    "huggingface_hub==0.25.2" \
    "einops" \
    "opencv-python" \
    "omegaconf" \
    "tqdm" \
    "pymeshlab" \
    "pygltflib" \
    "xatlas" \
    "rembg" \
    "onnxruntime" \
    "matplotlib" \
    2>&1 | tail -3

# Verify
!python3 -c "import torch; print('torch:', torch.__version__)"
!python3 -c "import torchvision; print('torchvision:', torchvision.__version__)"
!python3 -c "import transformers; print('transformers:', transformers.__version__)"
!python3 -c "import numpy; print('numpy:', numpy.__version__)"

print("\nDone. RESTART THE KERNEL, then run Steps 1, 2, 3 (clone only), skip this cell, continue from Step 4.")

In [ ]:
# Verify key package versions
import numpy, scipy, trimesh, transformers, diffusers
print(f"numpy: {numpy.__version__}")
print(f"scipy: {scipy.__version__}")
print(f"trimesh: {trimesh.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"diffusers: {diffusers.__version__}")

In [ ]:
# Build texture generation C++ extensions
!cd /workspace/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer && python3 setup.py install 2>&1 | tail -3
!cd /workspace/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer && python3 setup.py install 2>&1 | tail -3
print("Build complete.")

## Step 4: Preview input image

Upload your image to `/workspace/Hunyuan3D-2/` using the Jupyter file browser (left sidebar upload button).

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

INPUT_IMAGE = "/workspace/Hunyuan3D-2/banana.jpg"
OUTPUT_PATH = "/workspace/Hunyuan3D-2/output/model.glb"

img = Image.open(INPUT_IMAGE)
plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.axis("off")
plt.title("Input Image")
plt.show()
print(f"Image size: {img.size}")

## Step 5: Generate 3D mesh (shape)

In [ ]:
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline

shape_pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained("tencent/Hunyuan3D-2")
mesh = shape_pipeline(image=INPUT_IMAGE)[0]
print(f"Mesh generated: {len(mesh.vertices)} vertices, {len(mesh.faces)} faces")

## Step 6: Apply texture

In [ ]:
from hy3dgen.texgen import Hunyuan3DPaintPipeline

paint_pipeline = Hunyuan3DPaintPipeline.from_pretrained("tencent/Hunyuan3D-2")
textured_mesh = paint_pipeline(mesh, image=INPUT_IMAGE)
print("Texture applied successfully.")

## Step 7: Export 3D model

In [ ]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
textured_mesh.export(OUTPUT_PATH)
print(f"3D model saved to: {OUTPUT_PATH}")
print(f"File size: {os.path.getsize(OUTPUT_PATH) / 1024 / 1024:.2f} MB")

## Step 8: Visualize

In [ ]:
import numpy as np
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

vertices = np.array(textured_mesh.vertices)
faces = np.array(textured_mesh.faces)

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection="3d")

max_faces = 5000
if len(faces) > max_faces:
    indices = np.random.choice(len(faces), max_faces, replace=False)
    sampled_faces = faces[indices]
else:
    sampled_faces = faces

poly3d = [[vertices[vert] for vert in face] for face in sampled_faces]
ax.add_collection3d(Poly3DCollection(poly3d, alpha=0.5, edgecolor="k", linewidths=0.1))

scale = vertices.ptp(axis=0).max() / 2
mid = vertices.mean(axis=0)
ax.set_xlim(mid[0] - scale, mid[0] + scale)
ax.set_ylim(mid[1] - scale, mid[1] + scale)
ax.set_zlim(mid[2] - scale, mid[2] + scale)
ax.set_title("Generated 3D Mesh Preview")
plt.tight_layout()
plt.show()

## Step 9: Download model

In [ ]:
from IPython.display import FileLink
FileLink(OUTPUT_PATH)

## Step 10: Free GPU memory

In [ ]:
del shape_pipeline, paint_pipeline
torch.cuda.empty_cache()
print(f"VRAM used after cleanup: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")